## Query Pubchem for CAS number

This section will us the CID values found in the previous cells to acquire a CAS number from Pubchem. Often a molecule will have multiple CAS numbers and the `get_CAS_from_CID()` function will gather only one of them. Users should be aware that this looks for two dashes (-) in the numbers it receives from Pubchem to identify the CAS number. This procedure could be improved by a more systematic way of determining whether the item received from Pubchem is actually a CAS number. The REST queries each take at least 200 ms.

__If you stop this cell while it is running, you will lose all of your progress towards acquiring vendors__

In [ ]:
# Get the full list of CIDs from the library
cids = [int(x) for x in df['CID'].to_list() if x != '']

# Get the total number of CIDs for tracking progress
total = len(cids)

# Keep a list of CIDs that have no vendors
no_vendor_cids = []

# Enumerate over all CIDs and look for vendors
for i, cid in enumerate(cids):
    print(f'Working on {i + 1} of {total} ({round((i + 1) / total * 100, 2)}%)')

    # Try to get the list of PubchemVendor objects
    try:
        cas = get_CAS_from_cid(cid)
    except urllib.error.HTTPError as e:
        print(f'Could not get CAS number from CID {cid}. ERROR: {e}')
        continue

    # Add the CID/VENDORS based on inchi_key
    df.loc[df['CID'].astype(int) == cid, 'CAS_NUMBER'] = str(cas)

    # Check how many instances of that CID are in the df
    #if df[df['CID'].astype(int) == cid].shape[0] != 1:
    #    print(df[df['CID'] == cid])
    #    print(f'WARNING: Found more than one CID for {cid}!')
    ## Add the CID/VENDORS based on inchi_key
    #df.loc[df['CID'].astype(int) == cid, 'CAS'] = str(cas)

# Save the file
df.to_csv('./FINAL_LIBRARY_CURATED_with_cas_numbers.csv', index=False)



## Query a Sigman Inventory Export for the CAS numbers

Export a full copy of the Sigman inventory in labsuit and point the inventory_spreadsheet variable to its path. This cell will create a slice of the dataframe that contains CAS numbers in both your curated library and the inventory spreadsheet.

In [ ]:
# Define the spreadsheet file
inventory_spreadsheet = Path('./data/Sigman-inventory-03-07-2024-example.xlsx')

# Read in the file (These settings should read the default format)
try:
    inventory = pd.read_excel(file, header=0, sheet_name='Chemical', engine='openpyxl')
except Exception: # This is required because I think labsuit is not zipping their xlsx files
    with open(inventory_spreadsheet, 'rb') as infile:
        inventory = pd.read_excel(infile, sheet_name='Chemical')

# Filter inventory by presence of CAS
inventory['CAS_NUMBER'] = inventory['CAS_NUMBER'].astype(str).apply(str.strip)
inventory = inventory[inventory['CAS_NUMBER'].astype(str).isin(df['CAS_NUMBER'])]

display(inventory)

# Save the owned molecules in the results folder
inventory.to_csv('./results/owned_molecules.csv', index=False)

## Query Pubchem for SMILES from CAS Number
As written, the dataframe created from your spreadsheet must have the column name `CAS_NUMBER` somewhere in it

In [ ]:

# Define the spreadsheet file
inventory_spreadsheet = Path('./haruka/CBr-inventory-01-29-2026.xlsx')
sheet_name = 'Chemical'

# Define a directory in which to store information
smiles_from_cas_dir = Path('./haruka/smiles_from_cas_raw_files/')
smiles_from_cas_dir.mkdir(exist_ok=True)

# Read in the file (These settings should read the default format)
if inventory_spreadsheet.suffix == '.xlsx':
    try:
        inventory = pd.read_excel(file, header=0, sheet_name=sheet_name, engine='openpyxl')
    except Exception: # This is required because I think labsuit is not zipping their xlsx files
        with open(inventory_spreadsheet, 'rb') as infile:
            inventory = pd.read_excel(infile, sheet_name=sheet_name)
else:
    inventory = pd.read_csv(inventory_spreadsheet, header=0)

# Get the full list of CAS from the library
CAS_NUMBERS = [str(x) for x in inventory['CAS_NUMBER'].to_list() if x != '']

# Get the total number of CAS for tracking progress

total = len(CAS_NUMBERS)
print(f'There are {total} CAS numbers to process.')
print(f'There are {len(list(set(CAS_NUMBERS)))} unique CAS numbers to process.')
#from collections import Counter
#pprint(Counter(CAS_NUMBERS))
#assert False

# Keep a list of CAS numbers for which we could get no SMILES
no_vendor_cids = []

# Enumerate over all CAS and look for vendors
for i, CAS in enumerate(CAS_NUMBERS):
    print(f'Working on {i + 1} of {total} ({round((i + 1) / total * 100, 2)}%)')

    cas_file = Path(smiles_from_cas_dir / f'{CAS}.txt')

    if cas_file.exists():

        with open(cas_file, 'r') as _:
            smiles = str(_.read())

        print(f'Found {CAS} in {cas_file.absolute()}\t{smiles}')
    else:

        # Try to get the list of PubchemVendor objects
        try:
            smiles = get_SMILES_from_CAS(CAS)
            with open(cas_file, 'w', encoding='utf-8') as o:
                o.write(smiles)
        except urllib.error.HTTPError as e:
            print(f'Could not get SMILES string from CAS {CAS}. ERROR: {e}')
            continue

    # Add the CID/VENDORS based on inchi_key
    inventory.loc[inventory['CAS_NUMBER'].astype(str) == CAS, 'SMILES'] = str(smiles)

# Save the file
inventory.to_excel('./haruka/CBr-inventory-01-29-2026_with_smiles.xlsx', index=False)



## Query Pubchem for SMILES

This section will us the CID values found in the previous cell to acquire the canonical SMILES Pubchem. The REST queries each take at least 200 ms.

In [ ]:
# Read in the CIDs file
df = pd.read_table('./data/cids_small.txt', header=0)

# Make sure CID is present, integer, and use it as the dataframe index
df = df.dropna(subset=['CID']).copy()
df['CID'] = df['CID'].astype(int)
df = df.set_index('CID', drop=True)

# Ensure SMILES column exists
if 'SMILES' not in df.columns:
    df['SMILES'] = None

# Make a list of CIDs to process
cids = df.index.unique().tolist()
total = len(cids)

# Initialize SQLite cache
db_path = Path('./results/cache/smiles_cache.sqlite')

commit_every = 10
writes_since_commit = 0

# Initialize SQLite cache and make the connection
with init_smiles_cache(db_path) as conn:
    for i, cid in enumerate(cids, start=1):

        # Try cache first
        smiles, status = get_cached_smiles(conn=conn, cid=cid)

        if status == 'ok' and smiles:
            # Cached successful result
            df.loc[cid, 'SMILES'] = smiles
            continue

        if status == 'missing':
            # Store in the DataFrame as None so that it's empty cell
            df.loc[cid, 'SMILES'] = None
            continue

        # Do nothing (retry on http_error)
        if status not in ['ok', 'missing', 'http_error', 'parse_error', None]:
            raise ValueError(f'Unexpected cache status "{status}"')

        print(f'Working on {i} of {total} ({round(i / total * 100, 2)}%)')

        try:
            smiles = get_smiles_from_cid(cid)
        except urllib.error.HTTPError as e:
            # Cache the failure so you can decide later whether to retry
            upsert_cached_smiles(conn, cid, smiles='', status='http_error')
            writes_since_commit += 1
            print(f'Could not get SMILES from CID {cid}. ERROR: {e}')
            continue

        # Normalize / handle empty results
        if smiles is None or str(smiles).strip() == '':
            upsert_cached_smiles(conn, cid, '', 'missing')

            # Store in the DataFrame as None so that it's empty cell
            df.loc[cid, 'SMILES'] = None

            writes_since_commit += 1
        else:
            smiles = str(smiles).strip()
            upsert_cached_smiles(conn, cid, smiles, 'ok')
            df.loc[cid, 'SMILES'] = smiles
            writes_since_commit += 1

        # Commit periodically for speed + crash-safety
        if writes_since_commit >= commit_every:
            conn.commit()
            writes_since_commit = 0

    # Final commit
    conn.commit()

# Write to CSV
df.to_csv('./data/cids_with_smiles_james.csv')
print(df)
